# nb_11 — FactSales

**Purpose:** Build `FactSales` by joining `fact_sales.csv` to all relevant dimension surrogate keys.

## FK Resolution Map
| Source Column | Dimension | Surrogate Key |
|---|---|---|
| customer_id | DimCustomer | customer_key |
| product_id | DimProduct | product_key |
| salesrep_id | DimSalesRep | salesrep_key |
| territory_id | DimTerritory | territory_id (natural) |
| order_date | DimDate | order_date_key |
| ship_date | DimDate | ship_date_key |

Partitioned by `order_year`. OPTIMIZE + ZORDER by `(order_date_key, customer_key)`.

**Prerequisites:** nb_01, nb_04, nb_05, nb_06, nb_07 must be run first.

In [ ]:
from pyspark.sql.functions import col, to_date, year as _year

# Load dimension surrogate key lookups
dim_customer  = spark.sql("SELECT customer_id, customer_key FROM DimCustomer WHERE is_current = true")
dim_product   = spark.sql("SELECT product_id, product_key FROM DimProduct WHERE is_current = true")
dim_salesrep  = spark.sql("SELECT salesrep_id, salesrep_key FROM DimSalesRep WHERE is_current = true")
dim_territory = spark.sql("SELECT territory_id FROM DimTerritory")

date_map = spark.sql("""
    SELECT date_key,
           CAST(CONCAT(SUBSTR(CAST(date_key AS STRING),1,4),'-',
                       SUBSTR(CAST(date_key AS STRING),5,2),'-',
                       SUBSTR(CAST(date_key AS STRING),7,2)) AS DATE) AS cal_date
    FROM DimDate
""")

print("Dimensions loaded.")

In [ ]:
# Read fact_sales CSV
df_sales = spark.read.csv(
    "Files/seed_data/fact_sales.csv",
    header=True,
    inferSchema=True
)

print(f"Sales rows: {df_sales.count()}")
df_sales.printSchema()

In [ ]:
# Parse date columns
df_sales = (
    df_sales
    .withColumn("order_date_parsed", to_date(col("order_date")))
    .withColumn("ship_date_parsed",  to_date(col("ship_date")))
)

# Resolve order_date_key
df_sales = df_sales.join(
    date_map.withColumnRenamed("cal_date","order_date_parsed").withColumnRenamed("date_key","order_date_key"),
    on="order_date_parsed", how="left"
)

# Resolve ship_date_key
df_sales = df_sales.join(
    date_map.withColumnRenamed("cal_date","ship_date_parsed").withColumnRenamed("date_key","ship_date_key"),
    on="ship_date_parsed", how="left"
)

# Resolve dimension surrogate keys
df_sales = (
    df_sales
    .join(dim_customer,  on="customer_id",  how="left")
    .join(dim_product,   on="product_id",   how="left")
    .join(dim_salesrep,  on="salesrep_id",  how="left")
)

# Add partition column
df_sales = df_sales.withColumn("order_year", _year(col("order_date_parsed")))

print("All joins complete.")

In [ ]:
# Write FactSales partitioned by order_year
(
    df_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("order_year")
    .saveAsTable("FactSales")
)

spark.sql("OPTIMIZE FactSales ZORDER BY (order_date_key, customer_key)")
print("FactSales written and optimized.")
spark.sql("SELECT COUNT(*) AS row_count FROM FactSales").show()
spark.sql("SELECT order_year, COUNT(*) AS cnt FROM FactSales GROUP BY order_year ORDER BY order_year").show()